In [1]:
import jax
import optax
import equinox

In [2]:
class Linear(equinox.Module):

    weight: jax.Array
    bias: jax.Array

    def __init__(self, input_size, output_size, key):
        wkey, _ = jax.random.split(key)

        self.weight = jax.random.normal(
            wkey,
            (input_size, output_size)
        )
        self.bias = jax.numpy.zeros(
            (output_size,)
        )
    def __call__(self, x):
        return jax.numpy.dot(x, self.weight) + self.bias

In [3]:
class RecurrentNeuralNetworkCell(equinox.Module):

    input_cell: Linear
    hidden_cell: Linear

    def __init__(self, input_size, hidden_size, key):
        ikey, hkey = jax.random.split(key)

        self.input_cell = Linear(input_size, hidden_size, ikey)
        self.hidden_cell = Linear(hidden_size, hidden_size, hkey)

    def __call__(self, x, h):
        return jax.nn.tanh(
            self.input_cell(x) + self.hidden_cell(h)
        )

In [4]:
class Encoder(equinox.Module):
    hidden_layer: RecurrentNeuralNetworkCell

    def __init__(self, input_size, hidden_size, key):
        self.hidden_layer = RecurrentNeuralNetworkCell(input_size, hidden_size, key)

    def __call__(self, x_seq, h_prev):
        def step_fn(h, x_t):
            h = self.hidden_layer(x_t, h)
            return h, h

        h_final, h_seq = jax.lax.scan(step_fn, h_prev, x_seq)
        return h_final, h_seq

In [5]:
class Decoder(equinox.Module):
    hidden_layer: RecurrentNeuralNetworkCell
    output_layer: Linear

    def __init__(self, input_size, hidden_size, output_size, key):
        hkey, okey = jax.random.split(key, 2)
        self.hidden_layer = RecurrentNeuralNetworkCell(input_size, hidden_size, hkey)
        self.output_layer = Linear(hidden_size, output_size, okey)

    def __call__(self, y_seq, h_prev):
        def step_fn(h, y_t):
            h = self.hidden_layer(y_t, h)
            out = self.output_layer(h)
            return h, out

        h_final, p_seq = jax.lax.scan(step_fn, h_prev, y_seq)
        return h_final, p_seq

In [6]:
class Seq2Seq(equinox.Module):
    encoder: Encoder
    decoder: Decoder

    def __init__(self, enc_in, enc_hidden, dec_in, dec_hidden, out_size, key):
        ekey, dkey = jax.random.split(key, 2)
        self.encoder = Encoder(enc_in, enc_hidden, ekey)
        self.decoder = Decoder(dec_in, dec_hidden, out_size, dkey)

    def __call__(self, x_seq, y_seq, h0):
        h_enc, _ = self.encoder(x_seq, h0)
        h_dec, y_pred = self.decoder(y_seq, h_enc)
        return h_dec, y_pred

In [7]:
x_seq = jax.random.normal(
    jax.random.PRNGKey(1),
    (5, 10)
)

y_seq = jax.random.normal(
    jax.random.PRNGKey(2),
    (4, 8)
)

target = jax.random.normal(
    jax.random.PRNGKey(3),
    (4, 8)
)

In [8]:
key = jax.random.PRNGKey(0)

model = Seq2Seq(
    enc_in=10,
    enc_hidden=32,
    dec_in=8,
    dec_hidden=32,
    out_size=8,
    key=key
)

In [9]:
h0 = jax.numpy.zeros((32,))

In [10]:
h_final, y_pred = model(
    x_seq,
    y_seq,
    h0
)

print(y_pred.shape)

(4, 8)


In [11]:
def loss_fn(model, x_seq, y_seq, target, h0):

    _, pred = model(
        x_seq,
        y_seq,
        h0
    )

    return jax.numpy.mean(
        (pred - target) ** 2
    )

In [12]:
optimizer = optax.adam(1e-3)

opt_state = optimizer.init(
    equinox.filter(model, equinox.is_array)
)

In [13]:
@equinox.filter_jit
def train_step(
    model,
    opt_state,
    x_seq,
    y_seq,
    target,
    h0
):

    loss, grads = equinox.filter_value_and_grad(
        loss_fn
    )(model, x_seq, y_seq, target, h0)

    updates, opt_state = optimizer.update(
        grads,
        opt_state
    )

    model = equinox.apply_updates(
        model,
        updates
    )

    return model, opt_state, loss

In [14]:
for epoch in range(100):

    model, opt_state, loss = train_step(
        model,
        opt_state,
        x_seq,
        y_seq,
        target,
        h0
    )

    if epoch % 10 == 0:
        print(epoch, loss)

0 25.103075
10 19.11613
20 16.64122
30 13.897357
40 10.522054
50 9.436523
60 8.297572
70 7.247365
80 6.3861494
90 5.6340322
